# Laya × Gemma: quando a IA pode decidir e quando precisa de revisão humana?

**Autora:** Isabel Gonçalves Cruz · Hacktoberfest 2026 (FLUSP / IME-USP), trilha Open-Source AI

Este notebook compara dois modelos abertos numa triagem de chamados de atendimento em português:

- **Laya** (`convaiinnovations/laya-multilingual`, código e pesos Apache 2.0): um modelo de decisão não autorregressivo. Ele recebe o texto e uma pergunta tipada e devolve uma distribuição de probabilidade sobre as categorias numa única passada, sem gerar texto.
- **Gemma 4** (Google, Apache 2.0): um modelo generativo, que recebe as mesmas categorias e responde com JSON.

A pergunta do estudo não é qual modelo "vence". É **em quais chamados cada abordagem pode decidir sozinha e quais precisam de uma pessoa**, com números medidos e limitações explícitas.

**Protocolo**
1. 180 chamados sintéticos rotulados, com partição fixa: `ajuste` (72) e `avaliacao` (108).
2. Tudo o que é escolhido (variante do prompt, temperatura, limiar de revisão) usa **só** a partição de ajuste.
3. A partição de avaliação roda uma única vez, com a configuração congelada.
4. A probabilidade do Laya, a probabilidade calculada dos rótulos no Gemma (logits) e a confiança que o Gemma escreve ficam em colunas separadas. A confiança escrita **não** é tratada como probabilidade.

**Para reproduzir:** rode as células em ordem. No Colab, use GPU (T4 serve; L4/A100 permitem o `gemma-4-E4B-it`). Para testar o pipeline sem baixar pesos, use `MODO_SIMULACAO = True`. Nesse modo, os resultados vão para `resultados/simulacao/` e **não são resultados do estudo**.

## 1. Configuração

In [ ]:
MODO_SIMULACAO = False          # True: testa o pipeline com modelos falsos, sem baixar pesos
ERRO_ALVO = 0.10                # erro máximo aceito entre os chamados automatizados (fixado antes de rodar)
GEMMA_MODELO = None             # None = escolher pelo hardware; ou ex.: "google/gemma-4-E2B-it"
GEMMA_DTYPE = "auto"            # "auto", "bfloat16", "float16" ou "float32"
CALCULAR_PROB_GEMMA = True      # calcula P(rótulo) pelos logits do Gemma (6 passadas extras por chamado)
LAYA_CHECKPOINT = "multilingual"
VERSAO_LAYA = "0.4.19"
REPO = "https://github.com/bellDataSc/hacktoberfest-FUSP-2026"

In [ ]:
import os, subprocess, sys
from pathlib import Path

def _achar_raiz():
    for p in [Path.cwd(), Path.cwd().parent]:
        if (p / "triagem.py").exists():
            return p
    destino = Path("/content/hacktoberfest-FUSP-2026")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(destino)], check=True)
    return destino

RAIZ = _achar_raiz()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)

if "google.colab" in sys.modules and not MODO_SIMULACAO:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"laya=={VERSAO_LAYA}",
                    "transformers>=5", "accelerate", "psutil"], check=True)
print("raiz do projeto:", RAIZ)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import triagem as tg

SAIDA = RAIZ / "resultados" / ("simulacao" if MODO_SIMULACAO else "")
SAIDA.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

def pct(x):
    return "—" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.1%}".replace(".", ",")

def ic(par):
    return f"[{pct(par[0])}; {pct(par[1])}]"

if MODO_SIMULACAO:
    display(Markdown("> ⚠️ **MODO SIMULAÇÃO** — os números abaixo vêm de modelos falsos e servem só para testar o código."))

## 2. Ambiente de execução

In [ ]:
ambiente = tg.info_ambiente()
ambiente["modo_simulacao"] = MODO_SIMULACAO
print(json.dumps(ambiente, indent=2, ensure_ascii=False))

## 3. Dados

Os chamados são sintéticos e foram escritos para este estudo. As regras de rotulagem e as limitações estão em `dados/GUIA_ROTULAGEM.md`. Os tipos separam o que se quer testar: `direto`, `negacao` (a negação decide o rótulo), `ambiguo` (há uma segunda leitura plausível) e `insuficiente`.

In [ ]:
dados = tg.carregar_dados()
ajuste = dados[dados.particao == "ajuste"].reset_index(drop=True)
avaliacao = dados[dados.particao == "avaliacao"].reset_index(drop=True)
gabarito = dict(zip(dados.texto, dados.categoria))
print(f"{len(dados)} chamados · ajuste {len(ajuste)} · avaliação {len(avaliacao)}")
pd.crosstab([dados.categoria], [dados.particao, dados.tipo])

In [ ]:
dados.groupby("tipo").head(2)[["id", "texto", "categoria", "tipo", "rotulo_alternativo"]]

## 4. Laya

O Router do Laya mandaria um texto curto em português para o checkpoint multilíngue. Aqui o checkpoint fica **fixado** (`model="multilingual"`), e a coluna `roteamento` confirma qual checkpoint respondeu.

Duas variantes são comparadas **só na partição de ajuste**:
- `canonica`: uma pergunta, opções na ordem padrão;
- `rotacoes`: seis cópias da pergunta com as opções rotacionadas (`option_order`), respondidas na mesma passada e promediadas. O README do Laya documenta viés de posição das opções, e esta variante tenta neutralizá-lo.

In [ ]:
with tg.MedidorMemoria() as mem_laya_carga:
    if MODO_SIMULACAO:
        laya_clf = tg.LayaSimulado(gabarito, acuracia=0.70, semente=1)
    else:
        laya_clf = tg.LayaClassificador(checkpoint=LAYA_CHECKPOINT)
meta_laya = laya_clf.metadados()
meta_laya["hub"] = (tg.info_modelo_hub(f"convaiinnovations/laya-{LAYA_CHECKPOINT}")
                    if not MODO_SIMULACAO else {})
meta_laya["memoria_carga"] = mem_laya_carga.resultado
print(json.dumps(meta_laya, indent=2, ensure_ascii=False, default=str))

In [ ]:
laya_ajuste = {v: tg.executar_laya(laya_clf, ajuste, v) for v in ["canonica", "rotacoes"]}
selecao_laya = pd.DataFrame([
    {"variante": v, "acuracia": tg.acuracia(ajuste.categoria, p.pred), "f1_macro": tg.f1_macro(ajuste.categoria, p.pred),
     "rotacoes_divergentes": int(p.rotacoes_divergentes.sum()), "lat_p50_ms": tg.resumo_latencia(p.latencia_s)["p50_ms"]}
    for v, p in laya_ajuste.items()])
# Empate (diferença < 1 chamado) fica com a variante mais simples
melhor = selecao_laya.sort_values("f1_macro", ascending=False).iloc[0]
VARIANTE_LAYA = "canonica" if abs(melhor.f1_macro - selecao_laya.set_index("variante").f1_macro["canonica"]) < 1e-9 else melhor.variante
print("variante escolhida no ajuste:", VARIANTE_LAYA)
selecao_laya

In [ ]:
with tg.MedidorMemoria() as mem_laya_inf:
    laya_aval = tg.executar_laya(laya_clf, avaliacao, VARIANTE_LAYA)
meta_laya["memoria_inferencia_avaliacao"] = mem_laya_inf.resultado
meta_laya["variante"] = VARIANTE_LAYA
print(laya_aval.roteamento.value_counts().to_dict(), mem_laya_inf.resultado)

In [ ]:
del laya_clf
tg.liberar_memoria()

## 5. Gemma

O checkpoint é escolhido pelo hardware: `gemma-4-E4B-it` em GPU com 22 GB ou mais, e `gemma-4-E2B-it` nos demais casos. A licença é lida do model card e gravada nos metadados. Se o checkpoint usado não declarar `apache-2.0` (por exemplo, um Gemma 3), a célula avisa, e os termos específicos dele devem ser citados no README.

A decodificação é gulosa, sem modo de raciocínio (`enable_thinking=False`). Duas variantes são comparadas no ajuste:
- `zero_shot`: só as categorias e suas descrições, as mesmas que o Laya recebe;
- `few_shot`: além disso, um exemplo `direto` por categoria, tirado da partição de ajuste. Esses exemplos ficam fora da pontuação do ajuste.

In [ ]:
escolha = {"modelo": GEMMA_MODELO, "motivo": "definido manualmente"} if GEMMA_MODELO else tg.escolher_checkpoint_gemma(ambiente)
print(escolha)
hub_gemma = tg.info_modelo_hub(escolha["modelo"]) if not MODO_SIMULACAO else {}
print(hub_gemma)
if not MODO_SIMULACAO and hub_gemma.get("licenca_model_card") != "apache-2.0":
    display(Markdown(f"> ⚠️ Licença declarada: **{hub_gemma.get('licenca_model_card')}**. Confira os termos antes de publicar."))

In [ ]:
with tg.MedidorMemoria() as mem_gemma_carga:
    if MODO_SIMULACAO:
        gemma_clf = tg.GemmaSimulado(gabarito, acuracia=0.75, semente=2)
    else:
        gemma_clf = tg.GemmaClassificador(escolha["modelo"], dtype=GEMMA_DTYPE)
meta_gemma = {**gemma_clf.metadados(), "escolha": escolha, "hub": hub_gemma,
              "memoria_carga": mem_gemma_carga.resultado}

# Teste rápido: a saída precisa ser JSON válido e as probabilidades, finitas
teste = gemma_clf.gerar(tg.mensagens_gemma("Fui cobrado duas vezes este mês."))
print(teste)
if CALCULAR_PROB_GEMMA:
    pr = gemma_clf.probabilidades_rotulos(tg.mensagens_gemma("Fui cobrado duas vezes este mês."))
    assert np.isfinite(pr["probs"]).all(), "probabilidades não finitas: tente GEMMA_DTYPE='bfloat16' ou 'float32'"
    print(dict(zip(tg.ROTULOS, pr["probs"].round(3))))

In [ ]:
exemplos_fs = (ajuste[ajuste.tipo == "direto"].sort_values("id").groupby("categoria").head(1)
               .set_index("categoria").loc[tg.ROTULOS[:-1]])
# sem_informacao não tem tipo direto: usa o primeiro 'insuficiente' do ajuste
extra = ajuste[ajuste.categoria == "sem_informacao"].sort_values("id").iloc[0]
EXEMPLOS = [(r.texto, c) for c, r in exemplos_fs.iterrows()] + [(extra.texto, "sem_informacao")]
ids_exemplos = set(exemplos_fs.id) | {extra.id}
ajuste_sem_ex = ajuste[~ajuste.id.isin(ids_exemplos)].reset_index(drop=True)
print("exemplos few-shot:", sorted(ids_exemplos))

gemma_ajuste = {
    "zero_shot": tg.executar_gemma(gemma_clf, ajuste, "zero_shot", (), CALCULAR_PROB_GEMMA),
    "few_shot": tg.executar_gemma(gemma_clf, ajuste_sem_ex, "few_shot", EXEMPLOS, CALCULAR_PROB_GEMMA),
}
def _sel(v, p):
    ref = ajuste.set_index("id").loc[p.id, "categoria"].to_numpy()
    return {"variante": v, "n": len(p), "acuracia": tg.acuracia(ref, p.pred), "f1_macro": tg.f1_macro(ref, p.pred),
            "saidas_invalidas": int((p.status_saida == "invalido").sum()),
            "lat_p50_ms": tg.resumo_latencia(p.latencia_s)["p50_ms"]}
selecao_gemma = pd.DataFrame([_sel(v, p) for v, p in gemma_ajuste.items()])
melhor = selecao_gemma.sort_values("f1_macro", ascending=False).iloc[0]
VARIANTE_GEMMA = "zero_shot" if abs(melhor.f1_macro - selecao_gemma.set_index("variante").f1_macro["zero_shot"]) < 1e-9 else melhor.variante
print("variante escolhida no ajuste:", VARIANTE_GEMMA)
selecao_gemma

In [ ]:
EX_GEMMA = EXEMPLOS if VARIANTE_GEMMA == "few_shot" else ()
with tg.MedidorMemoria() as mem_gemma_inf:
    gemma_aval = tg.executar_gemma(gemma_clf, avaliacao, VARIANTE_GEMMA, EX_GEMMA, CALCULAR_PROB_GEMMA)
meta_gemma.update({"variante": VARIANTE_GEMMA, "exemplos_few_shot": sorted(ids_exemplos) if EX_GEMMA else [],
                   "memoria_inferencia_avaliacao": mem_gemma_inf.resultado})
print(gemma_aval.status_saida.value_counts().to_dict(), mem_gemma_inf.resultado)

In [ ]:
del gemma_clf
tg.liberar_memoria()

## 6. Comparação na partição de avaliação

Os intervalos são de 95%, por bootstrap com 2.000 reamostragens. A diferença entre modelos é pareada: os mesmos chamados são reamostrados para os dois. O teste de McNemar exato compara os chamados em que só um dos modelos acerta. Saídas inválidas do Gemma contam como erro.

In [ ]:
y = avaliacao.categoria.to_numpy()
assert (laya_aval.id.to_numpy() == avaliacao.id.to_numpy()).all() and (gemma_aval.id.to_numpy() == avaliacao.id.to_numpy()).all()
pl, pg = laya_aval.pred.to_numpy(), gemma_aval.pred.to_numpy()

linhas = [tg.resumo_classificacao(y, pl, f"Laya ({VARIANTE_LAYA})"),
          tg.resumo_classificacao(y, pg, f"Gemma ({VARIANTE_GEMMA}, resposta gerada)")]
if CALCULAR_PROB_GEMMA:
    linhas.append(tg.resumo_classificacao(y, gemma_aval.pred_prob.to_numpy(), "Gemma (argmax da prob. calculada)"))
lat = {"Laya": tg.resumo_latencia(laya_aval.latencia_s), "Gemma": tg.resumo_latencia(gemma_aval.latencia_s)}

tabela = pd.DataFrame([{
    "modelo": r["modelo"], "acurácia": pct(r["acuracia"]), "IC 95%": ic(r["acuracia_ic95"]),
    "F1 macro": pct(r["f1_macro"]), "IC 95% (F1)": ic(r["f1_macro_ic95"])} for r in linhas])
dif_acc = tg.bootstrap_diferenca(y, pl, pg, tg.acuracia)
dif_f1 = tg.bootstrap_diferenca(y, pl, pg, tg.f1_macro)
mcn = tg.mcnemar_exato(pl == y, pg == y)
display(tabela)
print(f"Laya − Gemma: acurácia {dif_acc['diferenca']:+.3f} IC95 {np.round(dif_acc['ic95'], 3)} · "
      f"F1 macro {dif_f1['diferenca']:+.3f} IC95 {np.round(dif_f1['ic95'], 3)}")
print(f"McNemar exato: só Laya acerta {mcn['so_a_acerta']}, só Gemma acerta {mcn['so_b_acerta']}, p = {mcn['p_valor']:.3f}")

In [ ]:
def _mem(m):
    return m.get("gpu_pico_mb", m.get("rss_delta_mb"))
custo = pd.DataFrame([
    {"modelo": "Laya", "checkpoint": meta_laya.get("checkpoint"), "device": meta_laya.get("device"),
     "precisão": meta_laya.get("dtype_autocast"), "latência p50 (ms)": round(lat["Laya"]["p50_ms"], 1),
     "latência p95 (ms)": round(lat["Laya"]["p95_ms"], 1),
     "memória pico carga+inferência (MB)": max(_mem(meta_laya["memoria_carga"]) or 0, _mem(meta_laya["memoria_inferencia_avaliacao"]) or 0)},
    {"modelo": "Gemma", "checkpoint": meta_gemma.get("modelo"), "device": meta_gemma.get("device"),
     "precisão": meta_gemma.get("dtype"), "latência p50 (ms)": round(lat["Gemma"]["p50_ms"], 1),
     "latência p95 (ms)": round(lat["Gemma"]["p95_ms"], 1),
     "memória pico carga+inferência (MB)": max(_mem(meta_gemma["memoria_carga"]) or 0, _mem(meta_gemma["memoria_inferencia_avaliacao"]) or 0)},
])
custo

**Como ler latência e memória.** Os dois modelos rodam um chamado por vez (lote 1), na mesma máquina, depois de 3 chamados de aquecimento. A latência do Gemma inclui a geração da resposta (`tokens_gerados` está nas predições), mas não as 6 passadas usadas para calcular a probabilidade dos rótulos, que ficam em `latencia_prob_s`. Na GPU, a memória é o pico de tensores do PyTorch (`max_memory_allocated`); na CPU, é o aumento do RSS do processo. As precisões numéricas diferem (o Laya usa o autocast do próprio pacote; o Gemma usa o `dtype` registrado), e isso faz parte das condições, não é um detalhe.

In [ ]:
por_tipo = avaliacao[["id", "tipo", "categoria"]].assign(laya=pl == y, gemma=pg == y)
resumo_tipo = por_tipo.groupby("tipo").agg(n=("id", "size"), laya=("laya", "mean"), gemma=("gemma", "mean"))
resumo_tipo.loc["contem_negacao (léxico)"] = [
    int(avaliacao.contem_negacao.sum()),
    por_tipo.laya[avaliacao.contem_negacao == 1].mean(),
    por_tipo.gemma[avaliacao.contem_negacao == 1].mean()]
resumo_tipo.style.format({"laya": "{:.1%}", "gemma": "{:.1%}", "n": "{:.0f}"})

In [ ]:
fig = tg.grafico_matrizes([("Laya", y, pl), ("Gemma (resposta gerada)", y, pg)])
fig.savefig(SAIDA / "matrizes_confusao.png", dpi=160, bbox_inches="tight")

In [ ]:
# Onde os modelos discordam, com foco em negação e ambiguidade
comp = avaliacao[["id", "texto", "categoria", "tipo", "rotulo_alternativo"]].assign(
    laya=pl, p_laya=laya_aval.p_max.round(3), gemma=pg, conf_escrita_gemma=gemma_aval.confianca_declarada)
comp[(comp.laya != comp.categoria) | (comp.gemma != comp.categoria)].sort_values("tipo")

## 7. Confiabilidade dos escores

Três sinais diferentes, tratados de formas diferentes:

| sinal | o que é | como é avaliado |
|---|---|---|
| Laya `p_max` | maior probabilidade da distribuição calculada pelo modelo | ECE, Brier e NLL, antes e depois de uma temperatura ajustada no ajuste |
| Gemma prob. calculada | softmax do log-prob de cada rótulo, dado o prompt e `{"categoria": "` | mesmas métricas, com temperatura própria |
| Gemma confiança escrita | número que o modelo escreve no JSON | **só** como sinal ordinal: AUROC (acerto × erro) e curva risco-cobertura |

Uma temperatura não muda qual categoria é escolhida, só a probabilidade associada a ela.

In [ ]:
y_aj_laya = ajuste.set_index("id").loc[laya_ajuste[VARIANTE_LAYA].id, "categoria"].to_numpy()
P_laya_aj = tg.matriz_probs(laya_ajuste[VARIANTE_LAYA])
T_LAYA = tg.ajustar_temperatura(P_laya_aj, tg.indices(y_aj_laya))
P_laya = tg.matriz_probs(laya_aval)
P_laya_cal = tg.aplicar_temperatura(P_laya, T_LAYA)

calib = [tg.resumo_calibracao(P_laya, y, "Laya bruto"),
         tg.resumo_calibracao(P_laya_cal, y, f"Laya com T={T_LAYA:.2f}")]
if CALCULAR_PROB_GEMMA:
    g_aj = gemma_ajuste[VARIANTE_GEMMA]
    y_aj_g = ajuste.set_index("id").loc[g_aj.id, "categoria"].to_numpy()
    T_GEMMA = tg.ajustar_temperatura(tg.matriz_probs(g_aj), tg.indices(y_aj_g))
    P_gemma = tg.matriz_probs(gemma_aval)
    P_gemma_cal = tg.aplicar_temperatura(P_gemma, T_GEMMA)
    calib += [tg.resumo_calibracao(P_gemma, y, "Gemma prob. calculada bruta"),
              tg.resumo_calibracao(P_gemma_cal, y, f"Gemma prob. calculada com T={T_GEMMA:.2f}")]
conf_escrita = gemma_aval.confianca_declarada.to_numpy(dtype=float)
calib.append({"sinal": "Gemma confiança escrita (ordinal)", "auroc_acerto": tg.auroc(conf_escrita, pg == y),
              "valores_distintos": int(pd.Series(conf_escrita).nunique()),
              "ausente": int(np.isnan(conf_escrita).sum())})
tabela_calib = pd.DataFrame(calib)
tabela_calib

O AUROC mede se o sinal ordena os acertos acima dos erros: 0,5 é acaso e 1,0 é separação perfeita. Ele é a única métrica que se aplica aos três sinais, porque não exige que o número seja uma probabilidade. Numa coluna em que todos os valores são iguais (por exemplo, sempre 0,9), o AUROC fica em 0,5: o sinal não informa nada.

In [ ]:
series = [{"rotulo": "Laya bruto", "conf": P_laya.max(1), "acerto": P_laya.argmax(1) == tg.indices(y), "cor": tg.COR_LAYA, "estilo": ":"},
          {"rotulo": f"Laya, T={T_LAYA:.2f}", "conf": P_laya_cal.max(1), "acerto": P_laya_cal.argmax(1) == tg.indices(y), "cor": tg.COR_LAYA}]
if CALCULAR_PROB_GEMMA:
    series.append({"rotulo": f"Gemma prob. calculada, T={T_GEMMA:.2f}", "conf": P_gemma_cal.max(1),
                   "acerto": P_gemma_cal.argmax(1) == tg.indices(y), "cor": tg.COR_GEMMA})
fig = tg.grafico_confiabilidade(series, titulo="Diagrama de confiabilidade · avaliação (bins com ≥ 3 chamados)")
fig.savefig(SAIDA / "confiabilidade.png", dpi=160, bbox_inches="tight")

## 8. Política de revisão humana para o fluxo do Laya

Regras, fixadas **antes** de olhar a avaliação:
1. Se a categoria prevista for `sem_informacao`, o chamado vai para uma pessoa, porque nenhuma equipe consegue agir sem falar com o cliente.
2. Nos demais casos, o chamado é automatizado se a probabilidade calibrada for maior ou igual ao limiar τ.
3. τ é o menor valor, **na partição de ajuste**, que mantém a taxa de erro entre os automatizados em no máximo `ERRO_ALVO`, com pelo menos 10 casos automatizados.

Com 72 chamados de ajuste, τ é uma estimativa ruidosa. Por isso a avaliação mostra intervalos e a curva completa.

In [ ]:
REVISAR_SEMPRE = ["sem_informacao"]
conf_aj = tg.aplicar_temperatura(P_laya_aj, T_LAYA).max(1)
escolha_tau = tg.escolher_limiar(conf_aj, laya_ajuste[VARIANTE_LAYA].pred, y_aj_laya, ERRO_ALVO, REVISAR_SEMPRE)
print("ajuste:", escolha_tau)

politica = tg.Politica(limiar=escolha_tau["limiar"], temperatura=T_LAYA, variante=VARIANTE_LAYA,
                       checkpoint=LAYA_CHECKPOINT, erro_alvo=ERRO_ALVO, revisar_sempre=REVISAR_SEMPRE,
                       observacao="SIMULAÇÃO" if MODO_SIMULACAO else "")
conf_av = P_laya_cal.max(1)
res_pol = tg.avaliar_politica(conf_av, pl, y, politica.limiar, REVISAR_SEMPRE)
res_pol.update(tg.bootstrap_politica(conf_av, pl, y, politica.limiar, REVISAR_SEMPRE))
print(json.dumps(res_pol, indent=2, ensure_ascii=False))

In [ ]:
display(Markdown(
    f"**Na avaliação ({res_pol['n']} chamados):** o Laya automatiza **{res_pol['automatizados']}** "
    f"({pct(res_pol['cobertura'])}, IC95 {ic(res_pol['cobertura_ic95'])}). Entre eles restam "
    f"**{res_pol['erros_entre_automatizados']} erros** ({pct(res_pol['taxa_erro_automatizados'])}"
    + (f", IC95 {ic(res_pol['taxa_erro_automatizados_ic95'])}" if res_pol['taxa_erro_automatizados_ic95'] else "")
    + f"). A revisão humana recebe {res_pol['enviados_revisao']} chamados, entre os quais "
    f"{res_pol['erros_capturados_pela_revisao']} dos {res_pol['erros_totais_sem_politica']} erros do modelo."))

In [ ]:
decisao = avaliacao[["id", "texto", "categoria", "tipo"]].assign(
    pred=pl, p_calibrada=conf_av.round(3),
    acao=np.where(~np.isin(pl, REVISAR_SEMPRE) & (conf_av >= politica.limiar), "automatico", "revisao"))
decisao["acertou"] = decisao.pred == decisao.categoria
display(pd.crosstab(decisao.tipo, [decisao.acao, decisao.acertou], margins=True))
print("Erros que passaram pela política (automatizados e errados):")
decisao[(decisao.acao == "automatico") & ~decisao.acertou]

In [ ]:
series_rc = [{"rotulo": "Laya (prob. calibrada)", "conf": conf_av, "acerto": pl == y, "cor": tg.COR_LAYA}]
if CALCULAR_PROB_GEMMA:
    series_rc.append({"rotulo": "Gemma (prob. calculada, calibrada)", "conf": P_gemma_cal.max(1),
                      "acerto": P_gemma_cal.argmax(1) == tg.indices(y), "cor": tg.COR_GEMMA})
series_rc.append({"rotulo": "Gemma (confiança escrita, ordinal)", "conf": conf_escrita, "acerto": pg == y,
                  "cor": tg.COR_GEMMA, "estilo": "--"})
fig = tg.grafico_risco_cobertura(series_rc, ERRO_ALVO, "Risco × cobertura · avaliação (sem a regra de sem_informacao)")
fig.savefig(SAIDA / "risco_cobertura.png", dpi=160, bbox_inches="tight")

A curva ignora a regra 1 e mostra só o efeito do sinal. Uma curva mais baixa significa menos erros para a mesma fração automatizada. Na curva da confiança escrita do Gemma, valores repetidos aparecem como degraus largos: o sinal só consegue separar os casos em poucos grupos.

In [ ]:
# Para comparação: a mesma regra aplicada ao Gemma (prob. calculada), com τ escolhido no ajuste dele
if CALCULAR_PROB_GEMMA:
    pred_g_aj = tg.aplicar_temperatura(tg.matriz_probs(g_aj), T_GEMMA)
    tau_g = tg.escolher_limiar(pred_g_aj.max(1), np.array(tg.ROTULOS)[pred_g_aj.argmax(1)], y_aj_g, ERRO_ALVO, REVISAR_SEMPRE)
    pred_g = np.array(tg.ROTULOS)[P_gemma_cal.argmax(1)]
    res_pol_gemma = {**tg.avaliar_politica(P_gemma_cal.max(1), pred_g, y, tau_g["limiar"], REVISAR_SEMPRE), "limiar": tau_g}
    print(json.dumps(res_pol_gemma, indent=2, ensure_ascii=False))

## 9. Exportação

In [ ]:
def _limpo(o):
    if isinstance(o, dict): return {k: _limpo(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpo(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    return o

pd.concat([laya_ajuste["canonica"].assign(particao="ajuste"), laya_ajuste["rotacoes"].assign(particao="ajuste"),
           laya_aval.assign(particao="avaliacao")]).to_csv(SAIDA / "predicoes_laya.csv", index=False)
pd.concat([gemma_ajuste["zero_shot"].assign(particao="ajuste"), gemma_ajuste["few_shot"].assign(particao="ajuste"),
           gemma_aval.assign(particao="avaliacao")]).to_csv(SAIDA / "predicoes_gemma.csv", index=False)
politica.salvar(SAIDA / "politica_laya.json")

metricas = {
    "ambiente": ambiente, "laya": meta_laya, "gemma": meta_gemma,
    "selecao_ajuste": {"laya": selecao_laya.to_dict("records"), "gemma": selecao_gemma.to_dict("records")},
    "classificacao_avaliacao": linhas, "latencia": lat,
    "diferenca_laya_menos_gemma": {"acuracia": dif_acc, "f1_macro": dif_f1, "mcnemar": mcn},
    "por_tipo": resumo_tipo.reset_index().to_dict("records"),
    "calibracao": tabela_calib.to_dict("records"),
    "politica_laya": {"config": politica.__dict__, "ajuste": escolha_tau, "avaliacao": res_pol},
    "politica_gemma_prob": res_pol_gemma if CALCULAR_PROB_GEMMA else None,
}
(SAIDA / "metricas.json").write_text(json.dumps(_limpo(metricas), indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(sorted(p.name for p in SAIDA.iterdir() if p.is_file()))

## 10. Resumo gerado a partir dos números desta execução

In [ ]:
L, G = linhas[0], linhas[1]
gpu = ambiente["gpu"]["nome"] if ambiente.get("gpu") else f"CPU ({ambiente['cpu']})"
texto = f'''
{"> ⚠️ SIMULAÇÃO: não publicar." if MODO_SIMULACAO else ""}
**Execução:** {ambiente["data_hora_utc"]} · {gpu} · laya {meta_laya.get("laya_versao")} ({meta_laya.get("checkpoint")}, variante `{VARIANTE_LAYA}`) · {meta_gemma.get("modelo")} ({meta_gemma.get("dtype")}, variante `{VARIANTE_GEMMA}`) · transformers {ambiente["pacotes"]["transformers"]}

- **Acurácia na avaliação (n={L["n"]}):** Laya {pct(L["acuracia"])} {ic(L["acuracia_ic95"])} · Gemma {pct(G["acuracia"])} {ic(G["acuracia_ic95"])}
- **F1 macro:** Laya {pct(L["f1_macro"])} · Gemma {pct(G["f1_macro"])}; diferença pareada {dif_f1["diferenca"]:+.3f}, IC95 [{dif_f1["ic95"][0]:+.3f}; {dif_f1["ic95"][1]:+.3f}]
- **Latência p50 (lote 1):** Laya {lat["Laya"]["p50_ms"]:.0f} ms · Gemma {lat["Gemma"]["p50_ms"]:.0f} ms
- **Calibração do Laya:** ECE {calib[0]["ece"]:.3f} bruto → {calib[1]["ece"]:.3f} com T={T_LAYA:.2f} (ajustada em {len(y_aj_laya)} chamados)
- **Política (erro-alvo {pct(ERRO_ALVO)}, τ={politica.limiar:.2f}):** {res_pol["automatizados"]}/{res_pol["n"]} automatizados ({pct(res_pol["cobertura"])}); {res_pol["erros_entre_automatizados"]} erros entre eles ({pct(res_pol["taxa_erro_automatizados"])})
'''
(SAIDA / "resumo.md").write_text(texto, encoding="utf-8")
display(Markdown(texto))

## 11. Limitações

- **Dados sintéticos e pequenos.** Os 180 chamados foram escritos para o estudo, são curtos e limpos, e os textos e os rótulos têm o mesmo autor. Com 108 chamados na avaliação, diferenças de poucos pontos ficam dentro do intervalo.
- **Zero-shot para os dois.** O Laya documenta que o ganho grande vem de fine-tuning no domínio. Aqui ele não foi ajustado. O Gemma também não.
- **Informação igual, interfaces diferentes.** Os dois recebem as mesmas categorias e descrições. A variante few-shot do Gemma recebe exemplos que o Laya não recebe, porque a pergunta tipada não tem esse canal.
- **Custo medido em uma máquina, lote 1.** O Laya responde em lote (`predict_batch`) e o Gemma pode rodar em servidores otimizados (vLLM, llama.cpp). Os números de latência e memória valem para esta configuração.
- **Probabilidades do Laya arredondadas.** O pacote devolve 4 casas decimais. A temperatura foi aplicada sobre `log(p)` com piso de 1e-6, sobre a temperatura do próprio checkpoint.
- **Temperatura e limiar com poucos dados.** Uma única temperatura e um único limiar foram ajustados em 72 chamados.
- **A probabilidade calculada do Gemma depende do formato.** Ela é condicionada ao prefixo `{"categoria": "` e normalizada sobre as 6 categorias. É uma propriedade do modelo, mas não é o que ele faria livremente.